# Packages

In [ ]:
! pip install datasets plotly nbformat seaborn matplotlib scipy numpy pandas scikit-learn -q

In [ ]:
import plotly.express as px
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


# Données

### Première méthode: A partir d'une API

In [1]:
import requests
import pandas as pd

url = "https://datasets-server.huggingface.co/rows"

all_rows = []

for offset in range(0, 1000, 100):
    params = {
        "dataset": "Uris001/credit-risk-eda",
        "config": "default",
        "split": "train",
        "offset": offset,
        "length": 100
    }

    response = requests.get(url, params=params).json()

    if "rows" not in response:
        break

    all_rows.extend([row["row"] for row in response["rows"]])

df = pd.DataFrame(all_rows)
print(df.head())

   Age  Annual Income Home Ownership  Employment Length (Years) Loan Purpose  \
0   21           9600            OWN                        5.0    EDUCATION   
1   25           9600       MORTGAGE                        1.0      MEDICAL   
2   23          65500           RENT                        4.0      MEDICAL   
3   24          54400           RENT                        8.0      MEDICAL   
4   21           9900            OWN                        2.0      VENTURE   

  Loan Grade  Loan Amount  Interest Rate  Default Status  Loan % of Income  \
0          B         1000          11.14               0              0.10   
1          C         5500          12.87               1              0.57   
2          C        35000          15.23               1              0.53   
3          C        35000          14.27               1              0.55   
4          A         2500           7.14               1              0.25   

  Previous Default  Credit History Length (Years) 

### Deuxième méthode

In [ ]:
from datasets import load_dataset

ds = load_dataset("Uris001/credit-risk-eda")

data = ds["train"].to_pandas()
print(data.shape)

## Affichage de la base de donnée

In [ ]:
data.head()

# Statistiques Descriptives

In [ ]:
data.info()

In [ ]:
data.describe()

In [ ]:
from statistique_descriptive import Statistique_Descriptives
from statistique_descriptive import analyse_categorielle


# I. Statistiques descriptives Univariée

### Statistiques descriptives de l'Age des clients

In [ ]:
Statistique_Descriptives(data, "Age")

Un premier control exploratoire montre que les données l'age des individus est dans une fourchette assez raisonnable comprise entre 20 et 84 ans. Cela semble très normale vu la situation des individus et l'esperance de vie à la naissance. 

### Analyse du nombre d'année de travail

In [ ]:
Statistique_Descriptives(data, "Employment Length (Years)")

Une analyse aussi de nombre d'année d'experience des individus montre que cela est aussi normale avec la periode allant de 0 à 41 ans d'experience de travail.

### Analyse des données des taux d'interet

In [ ]:
Statistique_Descriptives(data, "Interest Rate")

Les informations sur les taux d'interet montre que le mode est obtenus pour des valeurs proches de 7,5% ce qui reflète reellement la nature des taux d'interet sur les prèts bancaires. 

### Données sur les montants empruntés

In [ ]:
Statistique_Descriptives(data, "Loan Amount")

Le montant maximal des montants prets est de 35000 euros.

### Statistique descriptives sur le statut des clients

In [ ]:
analyse_categorielle(data, "Default Status")

Il existe environ 21,5 % des clients de la banque qui sont en defaut de paiement de leur pret.

### Analyse des defaut précedement des clients

In [ ]:
analyse_categorielle(data, "Previous Default")

Seulement 17,5% des clients avait fait un defaut sur leur credit precedemment.

### Analyse de Loan-to-Income Group (Montant du pret/Revenu)

In [ ]:
analyse_categorielle(data, "LTI Group")

### Loan Grade

In [ ]:
analyse_categorielle(data, "Loan Grade")

# Statistique descriptives bivariés

In [ ]:
from statistique_descriptive import bivarié_cat_num
from statistique_descriptive import bivarié_cat_cat
from scipy import stats

### Income vs Grade

In [ ]:
bivarié_cat_num(data, "Loan Grade", "Annual Income")

### LTI vs Grade

In [ ]:
bivarié_cat_cat(data, "Loan Grade", "LTI Group")

### Employment Length vs Grade

In [ ]:
bivarié_cat_num(data, "LTI Group","Employment Length (Years)")


## Correlation des variables importantes avec l'échellon du pret

In [ ]:
grade_numeric = {'A': 1, 'B': 2, 'C': 3, 'D': 4, 'E': 5, 'F': 6, 'G': 7}
data['grade_numeric'] = data['Loan Grade'].map(grade_numeric)
features = {
    'Annual Income': 'Income',
    'Loan % of Income': 'LTI',
    'Employment Length (Years)': 'Employment Length',
    'Credit History Length (Years)': 'Credit History'
}

print(f"{'Variable':<25} {'Correlation avec Loan Grade':<25} {'R²':<10}")
print("-" * 60)

for col, label in features.items():
    r, p = stats.pearsonr(data['grade_numeric'], data[col])
    print(f"{label:<25} {r:<25.4f} {r**2:<10.4f}")

prev_numeric = (data['Previous Default'] == 'Yes').astype(int)
r, p = stats.pearsonr(data['grade_numeric'], prev_numeric)
print(f"{'Previous Default':<25} {r:<25.4f} {r**2:<10.4f}")

### Analyse de la raison des prets

In [ ]:

plt.figure(figsize=(10,6))

order = data['Loan Purpose'].value_counts().index

sns.countplot(
    data=data,
    x='Loan Purpose',
    order=order,
    palette='Blues_r'
)

plt.title('Loan Purpose Distribution')
plt.xlabel('Loan Purpose')
plt.ylabel('Number of Borrowers')

plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig("loan_purpose_distribution.png", bbox_inches='tight', dpi=300)
plt.show()


On voit que la plus grande proportions des prets des individus sont pour financer l'éducation.

### Analyse du defaut de paiement selon les differentes raison des prets

In [ ]:
ct = pd.crosstab(
    data['Loan Purpose'],
    data['Default Status'],
    normalize='index'
)

ct = ct.loc[order]

ct.plot(
    kind='bar',
    stacked=True,
    figsize=(10,6),
    colormap='coolwarm'
)

plt.title('Default Rate by Loan Purpose')
plt.ylabel('Proportion')
plt.xlabel('Loan Purpose')
plt.xticks(rotation=30, ha='right')
plt.legend(['No Default', 'Default'])
plt.tight_layout()
plt.savefig("default_rate_by_loan_purpose.png", bbox_inches='tight', dpi=300)
plt.show()


### Analyse du taux de selon le statut de logement des emprunteurs.

In [ ]:
plt.figure(figsize=(7,5))

ax = sns.barplot(
    x='Home Ownership',
    y='Default Status',
    data=data,
    order=['RENT', 'OWN', 'MORTGAGE'],
    palette=['#F44336', '#2196F3', '#FFC107'],
    errorbar=None
)

# Add labels
for p in ax.patches:
    ax.annotate(f'{p.get_height():.2f}',
                (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='bottom', fontsize=10)

plt.title('Default Rate en fonction Home Ownership')
plt.xlabel('Home Ownership Status')
plt.ylabel('Default Probability')

plt.savefig("home_ownership_plot.png", bbox_inches='tight', dpi=300)
plt.show()


On remarque que la base de donnée est très deséquilibré comme on peut le voir ici environ 18% des prets des individus ne sont pas remboursées

In [ ]:
data['Previous Default'].value_counts(normalize=True) * 100


### Defaut de paiement en fonction des précedents défauts

In [ ]:
sns.barplot(x='Previous Default', y='Default Status', data=data, errorbar=None, color='#9C27B0')
plt.title('Default Rate by Previous Default History')
plt.savefig("previous_default_plot.png", bbox_inches='tight', dpi=300)
plt.show()

On voit ainsi que la situation de defaut precedent d'un client constitue une variable très importante pour anticiper si un individu paiera sa dette ou non.

### Default rate en fonction de variables importantes

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Create LTI Group before using it
data['LTI Group'] = pd.qcut(data['Loan % of Income'], q=5, labels=["Very Low","Low","Medium","High","Very High"])

# Create Income Group before using it
data['Income Group'] = pd.qcut(data['Annual Income'], q=5, labels=['Very Low', 'Low', 'Medium', 'High', 'Very High'])

# 1. Previous Default x Loan Grade
grade_order = ['A', 'B', 'C', 'D', 'E', 'F', 'G']
grouped1 = (data.groupby(['Loan Grade', 'Previous Default'])['Default Status']
              .mean()
              .reset_index()
              .rename(columns={'Default Status': 'default_rate'}))

sns.barplot(data=grouped1, x='Loan Grade', y='default_rate',
            hue='Previous Default', order=grade_order,
            ax=axes[0], palette=['steelblue', 'coral'])
axes[0].set_title('Default Rate by Grade fonction Previous Default')
axes[0].set_xlabel('Loan Grade')
axes[0].set_ylabel('Default Rate')
axes[0].set_ylim(0, 1)

# 2. Previous Default x LTI Group
grouped2 = (data.groupby(['LTI Group', 'Previous Default'])['Default Status']
              .mean()
              .reset_index()
              .rename(columns={'Default Status': 'default_rate'}))
lti_order = ['Very Low', 'Low', 'Medium', 'High', 'Very High']
sns.barplot(data=grouped2, x='LTI Group', y='default_rate',
            hue='Previous Default', order=lti_order,
            ax=axes[1], palette=['steelblue', 'coral'])
axes[1].set_title('Default Rate by LTI fonction Previous Default')
axes[1].set_xlabel('LTI Group')
axes[1].set_ylabel('Default Rate')
axes[1].set_ylim(0, 1)

# 3. Previous Default x Income Group
grouped3 = (data.groupby(['Income Group', 'Previous Default'])['Default Status']
              .mean()
              .reset_index()
              .rename(columns={'Default Status': 'default_rate'}))
income_order = ['Very Low', 'Low', 'Medium', 'High', 'Very High']
sns.barplot(data=grouped3, x='Income Group', y='default_rate',
            hue='Previous Default', order=income_order,
            ax=axes[2], palette=['steelblue', 'coral'])
axes[2].set_title('Default Rate by Income fonction Previous Default')
axes[2].set_xlabel('Income Group')
axes[2].set_ylabel('Default Rate')
axes[2].set_ylim(0, 1)

plt.tight_layout()
plt.show()

### Default rate versus Age Group

In [ ]:
bivarié_cat_cat(data, "Age Group", "Default Status")

On remarque que les groupes d'age les plus en defaut de paiements sont les plus jeunes agée de 18 à 25 ans.

### Structure de la correlation

In [ ]:
# This heatmap visualizes the correlation between all numeric features.
plt.figure(figsize=(10,6))
sns.heatmap(data.corr(numeric_only=True), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Matrix of Numeric Features')
plt.savefig("correlation_heatmap.png", bbox_inches='tight', dpi=300)
plt.show()

Comme nous l'avons précedemment vu, la nature du credit précedent de la personne influence grandement s'il sera en defaut ou non.

## Pretraitement de la base de donnée

### Encodage ordinale pour la variable Loan Grade

In [ ]:
grade_map = {'A':1,'B':2,'C':3,'D':4,'E':5}
data["Loan Grade"] = data["Loan Grade"].map(grade_map)

### Utilisation de One-Hot-Encoding pour les variables suivantes
- Home Ownership
- Loan Purpose

In [ ]:
#data = pd.get_dummies(data, columns=["Home Ownership","Loan Purpose"], drop_first=True)

### Mapping de Previous Default

In [ ]:
data["Previous Default"] = data["Previous Default"].map({"Yes":1,"No":0})

## Creation du modèle

In [ ]:
from sklearn.model_selection import train_test_split

X = data.drop(columns=["Default Status"])
y = data["Default Status"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

In [ ]:
print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

### Fonction de scoring

$$
WoE_i = \ln \left( \frac{ \frac{n_i^{(non\_defaut)}}{N^{(non\_defaut)}} }{ \frac{n_i^{(defaut)}}{N^{(defaut)}} } \right)
$$

- WoE > 0  → catégorie "bonne" (plus de non-défauts)
- WoE < 0  → catégorie "risquée" (plus de défauts)
- WoE ≈ 0  → neutre

In [ ]:
target = "Default Status"
features = [
    "Home Ownership",
    "Loan Purpose",
    "Loan Grade",
    "LTI Group",
    "Income Group",
    "Age Group",
    "Interest Rate Group",
    "Previous Default",
    "Credit History Length (Years)"
]

In [ ]:
def fonction_scoring(X, y, col):
    tab = pd.crosstab(X[col], y)
    
    tab.columns = ['non_def', 'def']
    
    tab['dist_non_def'] = tab['non_def'] / tab['non_def'].sum()
    tab['dist_def'] = tab['def'] / tab['def'].sum()
    
    tab['WoE'] = np.log(tab['dist_non_def'] / tab['dist_def'])
    tab['IV'] = (tab['dist_non_def'] - tab['dist_def']) * tab['WoE']
    
    return tab, tab['IV'].sum()

In [ ]:
X_train_woe = pd.DataFrame()
X_test_woe = pd.DataFrame()

In [ ]:
woe_maps = {}

for col in features:
    table, _ = fonction_scoring(X_train, y_train, col)
    
    # sauver le mapping WoE
    woe_maps[col] = table['WoE']
    
    # appliquer transformation
    X_train_woe[col] = X_train[col].map(table['WoE'])

In [ ]:
for col in features:
    table = fonction_scoring(X_train, y_train, col)[0]
    X_test_woe[col] = X_test[col].map(table['WoE'])

In [ ]:
X_train_woe = X_train_woe.fillna(0)
X_test_woe = X_test_woe.fillna(0)

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()

model.fit(X_train_woe, y_train)

### Prediction du score

In [ ]:
y_proba = model.predict_proba(X_test_woe)[:,1]

### Transformation en score de defaut

In [ ]:
score = 300 + (score - score.min()) * 550 / (score.max() - score.min())

In [ ]:
score

In [ ]:
import matplotlib.pyplot as plt

plt.hist(score, bins=30)